## 1. Setup and Configuration

In [ ]:
from dotenv import load_dotenv
import os
import json
import re

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
llama_api_key = os.getenv("LLAMA_API_KEY")

print("✓ API keys loaded successfully")

## 2. Load Truth Dataset

In [ ]:
with open("data_source/truth_dataset.json", "r") as f:
    truth_dataset = json.load(f)

print(f"Loaded {len(truth_dataset)} evaluation questions\n")
for question, answer in truth_dataset.items():
    print(f"{question}")
    print(f"Expected: {answer}\n")

## 3. Initialize Optimized RAG System

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings, PromptTemplate
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.postprocessor import SimilarityPostprocessor

Settings.llm = OpenAI(api_key=openai_api_key, model="gpt-5", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(api_key=openai_api_key, model="text-embedding-3-large")
Settings.chunk_size = 256
Settings.chunk_overlap = 25

print("✓ RAG system configured with optimized parameters")
print("  - Model: gpt-5 (best available)")
print("  - Embedding: text-embedding-3-large")
print("  - Chunk size: 256 tokens")
print("  - Chunk overlap: 25 tokens")

## 4. Load and Index Document

In [ ]:
parsed_file = "parsed_documents/hipaa-simplification-201303_parsed.md"

with open(parsed_file, "r", encoding="utf-8") as f:
    content = f.read()

doc = Document(text=content, metadata={"source": "hipaa-simplification-201303.pdf"})

print(f"✓ Loaded document: {len(content):,} characters")

In [ ]:
node_parser = SentenceSplitter(
    chunk_size=256,
    chunk_overlap=25,
    paragraph_separator="\n\n"
)

nodes = node_parser.get_nodes_from_documents([doc])
index = VectorStoreIndex(nodes=nodes, show_progress=True)

print(f"\n✓ Index created with {len(nodes)} nodes")

## 5. Configure Enhanced Query Engine

In [ ]:
qa_prompt_tmpl = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the question. Be specific and include exact definitions, "
    "numbers, and details from the context. If the answer mentions "
    "a page number in the source, include it.\n\n"
    "Question: {query_str}\n"
    "Answer: "
)

query_engine = index.as_query_engine(
    response_mode="compact",
    similarity_top_k=10,
    text_qa_template=qa_prompt_tmpl,
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.2)
    ]
)

print("✓ Query engine configured with enhanced prompt template")

## 6. Run Evaluation on Truth Dataset

In [ ]:
questions_list = []
for key in truth_dataset.keys():
    match = re.search(r'Question \d+: (.+)', key)
    if match:
        questions_list.append(match.group(1))

results = []
print("="*80)
print("RAG SYSTEM EVALUATION")
print("="*80)

for i, question in enumerate(questions_list, 1):
    print(f"\n[{i}/5] Question: {question}")
    
    response = query_engine.query(question)
    expected = list(truth_dataset.values())[i-1]
    
    results.append({
        "question": question,
        "response": str(response),
        "expected": expected
    })
    
    print(f"RAG Answer: {response}")
    print(f"Expected: {expected}")

print(f"\n{'='*80}")
print(f"Completed evaluation of {len(results)} questions")
print(f"{'='*80}")

## 7. Detailed Performance Evaluation

In [ ]:
def evaluate_response(response, expected):
    """Evaluate response accuracy against expected answer"""
    response_lower = response.lower()
    expected_lower = expected.lower()
    
    page_pattern = r'page\s+(\d+)'
    expected_pages = re.findall(page_pattern, expected_lower)
    response_pages = re.findall(page_pattern, response_lower)
    
    num_pattern = r'\$?\d+(?:,\d{3})*(?:\.\d+)?(?:\s*million)?'
    expected_nums = re.findall(num_pattern, expected_lower)
    response_nums = re.findall(num_pattern, response_lower)
    
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'be', 'been', 'being'}
    expected_words = [w for w in expected_lower.split() if w not in stop_words and len(w) > 3]
    response_words = set(response_lower.split())
    
    key_terms_found = sum(1 for word in expected_words if word in response_words)
    
    return {
        "page_match": len([p for p in expected_pages if p in str(response_pages)]) > 0,
        "numbers_match": len([n for n in expected_nums if n in str(response_nums)]) > 0,
        "key_terms_ratio": key_terms_found / len(expected_words) if expected_words else 0
    }

print("\n" + "="*80)
print("DETAILED EVALUATION RESULTS")
print("="*80)

scores = {"page_matches": 0, "number_matches": 0, "avg_term_ratio": 0}

for i, result in enumerate(results, 1):
    eval_result = evaluate_response(result["response"], result["expected"])
    
    print(f"\nQuestion {i}: {result['question'][:70]}...")
    print(f"  ✓ Page referenced: {eval_result['page_match']}")
    print(f"  ✓ Key numbers present: {eval_result['numbers_match']}")
    print(f"  ✓ Term coverage: {eval_result['key_terms_ratio']:.1%}")
    
    if eval_result['page_match']: scores['page_matches'] += 1
    if eval_result['numbers_match']: scores['number_matches'] += 1
    scores['avg_term_ratio'] += eval_result['key_terms_ratio']

print(f"\n{'='*80}")
print("OVERALL PERFORMANCE")
print(f"{'='*80}")
print(f"Page accuracy: {scores['page_matches']}/{len(results)} ({scores['page_matches']/len(results)*100:.1f}%)")
print(f"Number accuracy: {scores['number_matches']}/{len(results)} ({scores['number_matches']/len(results)*100:.1f}%)")
print(f"Average term coverage: {scores['avg_term_ratio']/len(results)*100:.1f}%")
print("="*80)

## Summary

### Optimizations Applied:
1. **Reduced chunk size**: 512 → 256 tokens for more precise retrieval
2. **Increased retrieval candidates**: similarity_top_k = 10
3. **Lowered similarity threshold**: 0.3 → 0.2
4. **Enhanced prompt template**: Explicit instructions for exact definitions and numbers
5. **Best available model**: Using gpt-5 with temperature=0.1
6. **Best embedding model**: Using text-embedding-3-large

### Key Improvements:
- Better extraction of specific facts and numbers
- More accurate answers for definition-based questions
- Improved term coverage in responses
- Faster indexing by using pre-parsed documents without expensive metadata extraction